In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import pandas as pd
from pathlib import Path

# Add src to path
sys.path.append('../src')

from utils.helpers import get_audio_files, format_time, print_banner

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Dataset Overview

In [ ]:
# Define data directory
DATA_DIR = '../data/raw'

# Get all raga folders
ragas = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]

print(f"Total Ragas: {len(ragas)}")
print(f"\nRagas found: {', '.join(ragas)}")

In [ ]:
# Count files per raga
dataset_stats = []

for raga in ragas:
    raga_path = os.path.join(DATA_DIR, raga)
    audio_files = get_audio_files(raga_path)
    dataset_stats.append({
        'Raga': raga,
        'File Count': len(audio_files)
    })

# Create DataFrame
df_stats = pd.DataFrame(dataset_stats)
print("\nDataset Statistics:")
print(df_stats)
print(f"\nTotal Files: {df_stats['File Count'].sum()}")

In [ ]:
# Visualize distribution
plt.figure(figsize=(10, 5))
plt.bar(df_stats['Raga'], df_stats['File Count'])
plt.xlabel('Raga')
plt.ylabel('Number of Files')
plt.title('Audio Files per Raga')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Audio File Analysis

In [ ]:
# Select a sample file for detailed analysis
sample_raga = ragas[0] if ragas else None

if sample_raga:
    sample_path = os.path.join(DATA_DIR, sample_raga)
    sample_files = get_audio_files(sample_path)
    
    if sample_files:
        sample_file = sample_files[0]
        print(f"Analyzing: {os.path.basename(sample_file)}")
        print(f"Raga: {sample_raga}")
    else:
        print("No audio files found!")
else:
    print("No raga folders found!")

In [ ]:
# Load audio file
if sample_file:
    y, sr = librosa.load(sample_file, duration=30)
    
    duration = len(y) / sr
    
    print(f"\nAudio Properties:")
    print(f"  Sample Rate: {sr} Hz")
    print(f"  Duration: {format_time(duration)}")
    print(f"  Samples: {len(y):,}")
    print(f"  Shape: {y.shape}")
    print(f"  Min value: {y.min():.4f}")
    print(f"  Max value: {y.max():.4f}")

## 3. Waveform Visualization

In [ ]:
# Plot waveform
plt.figure(figsize=(14, 4))
librosa.display.waveshow(y, sr=sr)
plt.title(f'Waveform - {sample_raga}')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## 4. Spectrogram Analysis

In [ ]:
# Compute mel spectrogram
mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

# Plot mel spectrogram
plt.figure(figsize=(14, 6))
librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel', fmax=8000)
plt.colorbar(format='%+2.0f dB')
plt.title(f'Mel Spectrogram - {sample_raga}')
plt.tight_layout()
plt.show()

In [ ]:
# Compute STFT spectrogram
D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)

# Plot
plt.figure(figsize=(14, 6))
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
plt.colorbar(format='%+2.0f dB')
plt.title(f'STFT Spectrogram - {sample_raga}')
plt.tight_layout()
plt.show()

## 5. Spectral Features

In [ ]:
# Extract spectral features
spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
zero_crossing_rate = librosa.feature.zero_crossing_rate(y)[0]

# Plot
frames = range(len(spectral_centroids))
t = librosa.frames_to_time(frames, sr=sr)

fig, ax = plt.subplots(3, 1, figsize=(14, 8))

ax[0].plot(t, spectral_centroids)
ax[0].set_ylabel('Spectral Centroid (Hz)')
ax[0].set_title('Spectral Centroid')
ax[0].grid(True)

ax[1].plot(t, spectral_rolloff, color='green')
ax[1].set_ylabel('Spectral Rolloff (Hz)')
ax[1].set_title('Spectral Rolloff')
ax[1].grid(True)

ax[2].plot(t, zero_crossing_rate, color='orange')
ax[2].set_ylabel('Zero Crossing Rate')
ax[2].set_xlabel('Time (s)')
ax[2].set_title('Zero Crossing Rate')
ax[2].grid(True)

plt.tight_layout()
plt.show()

## 6. Batch Audio Statistics

In [ ]:
# Analyze multiple files
audio_stats = []

for raga in ragas[:3]:  # Analyze first 3 ragas
    raga_path = os.path.join(DATA_DIR, raga)
    audio_files = get_audio_files(raga_path)[:5]  # First 5 files per raga
    
    for audio_file in audio_files:
        try:
            y, sr = librosa.load(audio_file, duration=30)
            duration = len(y) / sr
            
            audio_stats.append({
                'Raga': raga,
                'File': os.path.basename(audio_file),
                'Duration (s)': duration,
                'Sample Rate': sr,
                'Samples': len(y)
            })
        except Exception as e:
            print(f"Error loading {audio_file}: {e}")

# Display statistics
df_audio = pd.DataFrame(audio_stats)
print("\nAudio File Statistics:")
print(df_audio)

print("\nSummary:")
print(df_audio.groupby('Raga')['Duration (s)'].describe())

## Conclusion

This notebook provided an overview of the raga audio dataset including:
- Dataset composition and distribution
- Audio file properties
- Waveform and spectrogram visualizations
- Spectral feature analysis

**Next Steps:**
- Proceed to `02_pitch_detection.ipynb` for pitch analysis
- Analyze note extraction in `03_note_extraction.ipynb`